In [ ]:
from sklearn.metrics import average_precision_score, ndcg_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json

In [ ]:
def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f) 
    return data

def read_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append(item)
    return data

def evaluate_similarity_matrices(y_true_matrix, y_score_matrix, invalid_indices):
    """
    Evaluates a continuous similarity matrix against a graded ground truth matrix (0-4).
    Uses binarized ground truth (>= 3) for PR-AUC, Precision, and Recall.
    Uses the original graded ground truth for nDCG.
    """
    N = y_true_matrix.shape[0]

    valid_mask = np.ones(N, dtype=bool)
    valid_mask[invalid_indices] = False

    # --- NEW: Create binarized version (>= 3 is positive) ---
    y_true_binary = (y_true_matrix >= 3).astype(int)

    # Apply valid mask to graded, binary, and score matrices
    y_true_sub = y_true_matrix[valid_mask][:, valid_mask]
    y_true_bin_sub = y_true_binary[valid_mask][:, valid_mask]
    y_score_sub = y_score_matrix[valid_mask][:, valid_mask]

    M = y_true_sub.shape[0]
    diag_mask = ~np.eye(M, dtype=bool)

    # Remove diagonal and reshape for all three
    y_true_2d = y_true_sub[diag_mask].reshape(M, M - 1)
    y_true_bin_2d = y_true_bin_sub[diag_mask].reshape(M, M - 1)
    y_score_2d = y_score_sub[diag_mask].reshape(M, M - 1)

    # Filter out queries (rows) that have no positive examples in the BINARY matrix
    # (Otherwise PR-AUC and Recall will throw division-by-zero errors)
    has_positives_mask = y_true_bin_2d.sum(axis=1) > 0
    
    y_true_valid = y_true_2d[has_positives_mask]          # Graded (for nDCG)
    y_true_bin_valid = y_true_bin_2d[has_positives_mask]  # Binary (for PR-AUC, P, R)
    y_score_valid = y_score_2d[has_positives_mask]

    # --- 2. CALCULATING METRICS ---

    # PR-AUC (Uses Binary)
    pr_aucs = [
        average_precision_score(t, s) 
        for t, s in zip(y_true_bin_valid, y_score_valid)
    ]
    mean_pr_auc = np.mean(pr_aucs)

    # Vectorized sorting for Top-K metrics
    sorted_indices = np.argsort(-y_score_valid, axis=1)

    # Extract the BINARY true labels of the top 3 and top 5 predicted documents
    top_3_true_bin = np.take_along_axis(y_true_bin_valid, sorted_indices[:, :3], axis=1)
    top_5_true_bin = np.take_along_axis(y_true_bin_valid, sorted_indices[:, :5], axis=1)

    # Total actual positives per row (Uses Binary)
    total_positives = y_true_bin_valid.sum(axis=1)

    # Precision @ 3 and 5 (Uses Binary)
    p_at_3 = np.mean(top_3_true_bin.sum(axis=1) / 3.0)
    p_at_5 = np.mean(top_5_true_bin.sum(axis=1) / 5.0)

    # Recall @ 3 and 5 (Uses Binary)
    r_at_3 = np.mean(top_3_true_bin.sum(axis=1) / total_positives)
    r_at_5 = np.mean(top_5_true_bin.sum(axis=1) / total_positives)

    # nDCG (Uses Graded)
    # scikit-learn's ndcg_score directly accepts the 0-4 scores and processes them
    mean_ndcg = ndcg_score(y_true_valid, y_score_valid)

    # --- 3. RESULTS ---
    results = {
        "PR-AUC": mean_pr_auc.item(),
        "Precision@3": p_at_3.item(),
        "Precision@5": p_at_5.item(),
        "Recall@3": r_at_3.item(),
        "Recall@5": r_at_5.item(),
        "nDCG": mean_ndcg
    }

    return results

In [ ]:
# GROUND_TRUTH COLLECTED IN THE EXPERIMENT
exp_results = read_json('exp_results_v7.json')
exp_results = pd.DataFrame(exp_results)

In [ ]:
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]
final_results = []

for n_material in range(course_materials.shape[0]):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    with open("./Agent_resources/Titles/pid{}_{}_title.txt".format(course_id,material_id), "r", encoding="utf-8") as f:
        lecture_title = f.read()
        
    print("ID: {}".format(material_id))
    print("N pages: {}".format(max_pages))
    print("Title: {}".format(title))
    print("Lecture Title: {}".format(lecture_title))

    slide_types = []
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())


    non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]

    grounds = np.zeros((max_pages,max_pages))
    non_info_filter = []
    possible_slides = []
    for i in range(1,max_pages+1):
        if not slide_types[i-1] in non_informative:
            possible_slides.append(i-1)
            grounds[i-1,i-1] = 4
            for j in range(i+1,max_pages+1):
                if not slide_types[j-1] in non_informative:
                    # print(f"{i} - {j}")
                    gt_ij_a = exp_results.loc[(exp_results["user"]==5626)&(exp_results["content_id"]==material_id)&(exp_results["slide_a"]==i)&(exp_results["slide_b"]==j)]
                    gt_ij_b = exp_results.loc[(exp_results["user"]==6817)&(exp_results["content_id"]==material_id)&(exp_results["slide_a"]==i)&(exp_results["slide_b"]==j)]
                    gt_ij_c = exp_results.loc[(exp_results["user"]==6850)&(exp_results["content_id"]==material_id)&(exp_results["slide_a"]==i)&(exp_results["slide_b"]==j)]
                    gt_ij_d = exp_results.loc[(exp_results["user"]==6886)&(exp_results["content_id"]==material_id)&(exp_results["slide_a"]==i)&(exp_results["slide_b"]==j)]

                    if gt_ij_a["relation"].values[0]==1 or gt_ij_a["reason"].values[0]!="":
                        # print(f"{i} - {j}: Relation")
                        grounds[i-1,j-1] += 1
                        grounds[j-1,i-1] += 1

                    if gt_ij_b.shape[0]>0 and (gt_ij_b["relation"].values[0]==1 or gt_ij_b["reason"].values[0]!=""):
                        # print(f"{i} - {j}: Relation")
                        grounds[i-1,j-1] += 1
                        grounds[j-1,i-1] += 1

                    if gt_ij_c.shape[0]>0 and (gt_ij_c["relation"].values[0]==1 or gt_ij_c["reason"].values[0]!=""):
                        # print(f"{i} - {j}: Relation")
                        grounds[i-1,j-1] += 1
                        grounds[j-1,i-1] += 1

                    if gt_ij_d.shape[0]>0 and (gt_ij_d["relation"].values[0]==1 or gt_ij_d["reason"].values[0]!=""):
                        # print(f"{i} - {j}: Relation")
                        grounds[i-1,j-1] += 1
                        grounds[j-1,i-1] += 1
        else:
            non_info_filter.append(i-1)
    
    # grounds = (grounds>1)*1

    # # Create the heatmap
    # plt.figure(figsize=[10,10])
    # ax = sns.heatmap(grounds,xticklabels=np.arange(1,max_pages+1),yticklabels=np.arange(1,max_pages+1),cmap=plt.get_cmap("viridis"),linewidth=.5,linecolor="black",vmin=0.4)

    # # Display the plot
    # plt.savefig(f"./Agent_resources/Similarities/ground_truth_{course_id}_{material_id}.png",dpi=300,bbox_inches="tight")

    bge = np.load(f'./Agent_resources/Similarities/bge_similiarity_{course_id}_{material_id}.npy')
    simcse = np.load(f'./Agent_resources/Similarities/simcse_similiarity_{course_id}_{material_id}.npy')
    simcse_seeker = np.load(f'./Agent_resources/Similarities/simcse_seeker_{course_id}_{material_id}.npy')
    nve = np.load(f'./Agent_resources/Similarities/nve_similiarity_{course_id}_{material_id}.npy')
    openai = np.load(f'./Agent_resources/Similarities/openai_similiarity_{course_id}_{material_id}.npy')
    kalm = np.load(f'./Agent_resources/Similarities/kalm_similiarity_{course_id}_{material_id}.npy')
    qwen = np.load(f'./Agent_resources/Similarities/qwen_similiarity_{course_id}_{material_id}.npy')

    vanilla_mean = np.load(f'./Agent_resources/Similarities/vanilla_mean_similiarity_{course_id}_{material_id}.npy')
    vanilla_seeker = np.load(f'./Agent_resources/Similarities/vanilla_seeker_similiarity_{course_id}_{material_id}.npy')
    ds_mean = np.load(f'./Agent_resources/Similarities/roberta_ds_mean_similiarity_{course_id}_{material_id}.npy')
    ds_seeker = np.load(f'./Agent_resources/Similarities/roberta_ds_seeker_similiarity_{course_id}_{material_id}.npy')
    ft_mean = np.load(f'./Agent_resources/Similarities/roberta_ds_ft_mean_similiarity_{course_id}_{material_id}.npy')
    ft_seeker = np.load(f'./Agent_resources/Similarities/roberta_ds_ft_seeker_similiarity_{course_id}_{material_id}.npy')

    fft_m2t1 = np.load(f'./Agent_resources/Similarities/m02_t01_{course_id}_{material_id}.npy')
    fft_m2t1_seeker = np.load(f'./Agent_resources/Similarities/m02_t01_seeker_{course_id}_{material_id}.npy')
    fft_m2t1_batch = np.load(f'./Agent_resources/Similarities/m02_t01_batch_{course_id}_{material_id}.npy')
    fft_m2t1_pre = np.load(f'./Agent_resources/Similarities/m02_t01_pre_{course_id}_{material_id}.npy')
    fft_m3t1 = np.load(f'./Agent_resources/Similarities/m03_t01_{course_id}_{material_id}.npy')

    ensemble = (openai + fft_m2t1_seeker)/2
    ensemble_2 = (openai + qwen)/2
    ensemble_3 = (openai + kalm)/2
    ensemble_4 = (openai + nve)/2
    ensemble_5 = (openai + bge)/2
    
    results = {}
    results["bge"] = evaluate_similarity_matrices(grounds,bge,non_info_filter)
    results["simcse"] = evaluate_similarity_matrices(grounds,simcse,non_info_filter)
    results["simcse_seeker"] = evaluate_similarity_matrices(grounds,simcse_seeker,non_info_filter)
    results["nve"] = evaluate_similarity_matrices(grounds,nve,non_info_filter)
    results["openai"] = evaluate_similarity_matrices(grounds,openai,non_info_filter)
    results["kalm"] = evaluate_similarity_matrices(grounds,kalm,non_info_filter)
    results["qwen"] = evaluate_similarity_matrices(grounds,qwen,non_info_filter)

    results["vanilla_mean"] = evaluate_similarity_matrices(grounds,vanilla_mean,non_info_filter)
    results["vanilla_seeker"] = evaluate_similarity_matrices(grounds,vanilla_seeker,non_info_filter)
    results["ds_mean"] = evaluate_similarity_matrices(grounds,ds_mean,non_info_filter)
    results["ds_seeker"] = evaluate_similarity_matrices(grounds,ds_seeker,non_info_filter)
    results["ft_mean"] = evaluate_similarity_matrices(grounds,ft_mean,non_info_filter)
    results["ft_seeker"] = evaluate_similarity_matrices(grounds,ft_seeker,non_info_filter)

    results["fft_m2t1"] = evaluate_similarity_matrices(grounds,fft_m2t1,non_info_filter)
    results["fft_m2t1_seeker"] = evaluate_similarity_matrices(grounds,fft_m2t1_seeker,non_info_filter)
    results["fft_m2t1_batch"] = evaluate_similarity_matrices(grounds,fft_m2t1_batch,non_info_filter)
    results["fft_m2t1_pre"] = evaluate_similarity_matrices(grounds,fft_m2t1_pre,non_info_filter)
    results["fft_m3t1"] = evaluate_similarity_matrices(grounds,fft_m3t1,non_info_filter)
    

    results["ensemble"] = evaluate_similarity_matrices(grounds,ensemble,non_info_filter)
    results["ensemble_2"] = evaluate_similarity_matrices(grounds,ensemble_2,non_info_filter)
    results["ensemble_3"] = evaluate_similarity_matrices(grounds,ensemble_3,non_info_filter)
    results["ensemble_4"] = evaluate_similarity_matrices(grounds,ensemble_4,non_info_filter)
    results["ensemble_5"] = evaluate_similarity_matrices(grounds,ensemble_5,non_info_filter)

    final_results.append(pd.DataFrame(results).T)

ID: 4eff0720836a198b6174eecf02cbfdbf
N pages: 28
Title: 第０回：授業の準備
Lecture Title: Session 0: Course Preparation
ID: 5a4be1fa34e62bb8a6ec6b91d2462f5a
N pages: 15
Title: 第７回：離散時間システム～線形時不変システム，差分方程式～
Lecture Title: Session 7: Discrete-Time Systems – Linear Time-Invariant Systems and Difference Equations
ID: 69f62956429865909921fa916d61c1f8
N pages: 19
Title: 第５回：離散時間信号とZ変換
Lecture Title: Lecture 5: Discrete-Time Signals and the Z-Transform
ID: 70f250e2d762fbde8a2e70eabf6eb953
N pages: 32
Title: 第１回：ディジタル信号処理の概要
Lecture Title: Overview of Digital Signal Processing
ID: 885b2c7a6deb4fea10f319c4ce993e02
N pages: 15
Title: 第９回：離散フーリエ変換
Lecture Title: Lecture 9: Discrete Fourier Transform
ID: 5516adb142fcb18a017c72602abbdb6d
N pages: 14
Title: 第２回：周期信号とフーリエ級数
Lecture Title: Session 2: Periodic Signals and Fourier Series
ID: 9161ab7a1b61012c4c303f10b4c16b2c
N pages: 14
Title: 第１０回：高速フーリエ変換
Lecture Title: Lecture 10: Fast Fourier Transform
ID: 81374713d991042a0e18865aa693cc24
N pages: 20
Title: 第

In [7]:
sum(final_results)/len(final_results)

,PR-AUC,Precision@3,Precision@5,Recall@3,Recall@5,nDCG
bge,0.775966,0.653634,0.540405,0.588436,0.773901,0.888450
simcse,0.687801,0.562009,0.472508,0.486148,0.647474,0.832897
simcse_seeker,0.719282,0.612371,0.507475,0.529428,0.716767,0.850799
nve,0.779265,0.682053,0.560912,0.603061,0.783904,0.888105
openai,0.804745,0.684955,0.561671,0.613703,0.797851,0.906763
kalm,0.776829,0.648696,0.550613,0.583817,0.779009,0.888876
qwen,0.786769,0.673052,0.571288,0.594736,0.802964,0.888121
vanilla_mean,0.577984,0.468898,0.405548,0.381620,0.539401,0.758207
vanilla_seeker,0.637186,0.533114,0.455117,0.450246,0.612452,0.794022
ds_mean,0.601728,0.509590,0.415657,0.420440,0.557210,0.773979
